In [ ]:
import os
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

base_dir = "/content/drive/MyDrive/EarthEngine_ML_Data"
l8_file = os.path.join(base_dir, "l8_hybrid_dswe_v11.csv")
rain_file = os.path.join(base_dir, "era5_daily_continuous_2015_2021_v2.csv")
output_file = os.path.join(base_dir, "raw_merged_dataset.csv")

print("Loading data...")
l8_df = pd.read_csv(l8_file)
rain_df = pd.read_csv(rain_file)

print("Building rainfall features...")
rain_df["Date"] = pd.to_datetime(rain_df["Date"])
rain_df["Region_ID"] = rain_df["Region_ID"].astype(int)
rain_df = rain_df.sort_values(["Region_ID", "Date"])
rain_df["Precip_mm"] = rain_df["Precip_mm"].clip(lower=0.0)

def build_rain_features(g):
    daily = g["Precip_mm"].shift(1)
    out = pd.DataFrame(index=g.index)

    for w in (7, 15, 30, 60, 90):
        out[f"rain_sum_{w}"] = daily.rolling(window=w, min_periods=1).sum()

    out["rain_days_7"] = (daily > 1.0).astype(float).rolling(window=7).sum()
    return out

rain_feats = rain_df.groupby("Region_ID").apply(build_rain_features).reset_index(level=0, drop=True)
rain_df = rain_df.join(rain_feats)

print("Merging...")
l8_df["Date"] = pd.to_datetime(l8_df["Date"])
l8_df["Region_ID"] = l8_df["Region_ID"].astype(int)

merged_df = pd.merge(l8_df, rain_df, on=["Date", "Region_ID"], how="left")

merged_df.to_csv(output_file, index=False)

print("-" * 30)
print(f"✅ Raw Data Merged & Saved: {output_file}")
print(f"   Total Rows: {len(merged_df)}")
print("-" * 30)


Mounted at /content/drive
Loading Data...
Calculating Rainfall History...
Merging Datasets...
------------------------------
✅ Raw Data Merged & Saved: /content/drive/MyDrive/EarthEngine_ML_Data/raw_merged_dataset.csv
   Total Rows: 1560
------------------------------


/tmp/ipython-input-904430491.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  lag_features = rain_df.groupby('Region_ID').apply(calculate_lags).reset_index(level=0, drop=True)
